# Hugging Face TGI experiment

A server for deploying and generating text from supported models. This notebook uses the same sample document and retrieval question as the cache-tool notebooks.

In [ ]:
# Setup: Start a compatible TGI server at localhost:8080, then run `pip install huggingface-hub` in the client environment.
# Client package: pip install huggingface-hub
print('Start the configured model server before running inference cells.')

## Using Hugging Face TGI for RAG and agentic AI

It provides a model endpoint for RAG answer generation and production inference features. The model engine manages its native cache; use an application cache for per-answer keys, TTL, and deletion.

In [ ]:
from huggingface_hub import InferenceClient
# Start TGI locally on port 8080 with a compatible model.
client = InferenceClient(base_url='http://localhost:8080')
def ask_rag(question, retrieved_context):
    response = client.chat.completions.create(messages=[{'role': 'system', 'content': f'Use this reference context to answer:\n{retrieved_context}'}, {'role': 'user', 'content': question}], max_tokens=120)
    return response.choices[0].message.content

## 1. RAG workflow

The notebook searches the shared sample document and sends the retrieved passage and question to this model tool.

In [ ]:
import re

sample_document = 'RAG retrieves relevant passages from documents before answering a question.\nA cache stores frequently requested RAG retrieval results so repeated questions return faster.\nAgent memory stores task progress and tool results so later steps can continue the task.\nRedis stores values by key and can expire them automatically using a time-to-live.'
document_version = "sample-doc-v1"
question = "How can a cache speed up repeated RAG retrieval?"

def retrieve_relevant_passage(question, document):
    stop_words = {"how", "can", "a", "the", "up", "to", "and", "for", "does"}
    query_terms = set(re.findall(r"[a-z0-9]+", question.lower())) - stop_words
    passages = [line.strip() for line in document.splitlines() if line.strip()]
    ranked = sorted(
        passages,
        key=lambda passage: len(
            query_terms & set(re.findall(r"[a-z0-9]+", passage.lower()))
        ),
        reverse=True,
    )
    if not ranked or not (
        query_terms & set(re.findall(r"[a-z0-9]+", ranked[0].lower()))
    ):
        raise ValueError("No matching passage was found in the sample document.")
    return ranked[0]

retrieved_context = retrieve_relevant_passage(question, sample_document)
answer = ask_rag(question, retrieved_context)
print("Question:", question)
print("Retrieved context:", retrieved_context)
print("Generated answer:", answer)

## 2. Agent session workflow

Reuse the retrieved context as part of the current agent request. Persistent session state is managed by your application.

In [ ]:
agent_question = input('Enter the agent question about the retrieved passage: ').strip()
agent_answer = ask_rag(agent_question, retrieved_context)
print('Agent answer:', agent_answer)

## 3. Prompt and response workflow

Send a prompt using the retrieved passage. The engine may reuse shared prefixes; response-key caching is not provided by its KV cache.

In [ ]:
prompt_question = input('Enter another question about the same passage: ').strip()
print('Model response:', ask_rag(prompt_question, retrieved_context))

## 4. TTL and freshness

The engine controls capacity and eviction for its KV/prefix cache. Refresh retrieved context when the source document changes.

In [ ]:
document_version = 'sample-doc-v1'
print('Use the current document version in your RAG application:', document_version)

## 5. Agent tool result workflow

A native model KV cache is not a key/value store for arbitrary tool results. Use an application cache for those values.

In [ ]:
tool_result = retrieved_context
print('Actual retrieved tool result to hand to the agent:', tool_result)

## 6. Read or reuse the result

Repeat a model request with the same retrieved context prefix to exercise the inference engine's native reuse path.

In [ ]:
repeat_question = input('Repeat or rephrase the question: ').strip()
print('Model response:', ask_rag(repeat_question, retrieved_context))

## 7. Invalidate stale data

Native KV entries are evicted by the engine. When the document changes, retrieve the updated passage before asking the model.

In [ ]:
updated_document = input('Paste the updated document text: ').strip()
updated_question = input('Question for the updated document: ').strip()
updated_context = retrieve_relevant_passage(updated_question, updated_document)
print('Updated model answer:', ask_rag(updated_question, updated_context))

## 8. Application workflow

Use this same flow in your application: retrieve from the current document/index version, build a versioned key, check this tool, and on a miss store the real retrieved context. Pass that context to your LLM for generation. Agent state uses a separate session namespace.

In [ ]:
print('Hugging Face TGI: sample-document RAG and agent cache example completed. Use README.md for when and why to choose this tool.')